# Fase 2 — Transformação dos Dados no Dataset
---

## Objetivo 

Identificar colunas que precisam de alterações

- Resolver colunas numéricas divididas em intervalos
- Tratar colunas com valores nulos em classifições
- Partir de colunas categóricas para valores numéricos
- Analisar interferência dos outliers
- Avaliar colunas com baixa variância
- Resolver colunas que são valores compostos (listas)
- Usar a coluna do CNAE para melhor identificação
- Criar colunas derivadas a partir de dados dos clientes: Feature engineering
- Padronizar/normalizar variáveis numéricas quando necessário
- Separar treino/teste antes de transformações críticas


## Datasets disponíveis

| Arquivo | Descrição |
|---|---|
| `credito_aplicacao_clientes.csv` | Dados cadastrais de 200 clientes (Serasa, iFood, Google Maps, etc.) |
| `credito_comportamental_pedidos.csv` | Histórico de 200 pedidos de clientes já ativos |

## 0. Importações

- `pandas`: manipulação e análise de dados tabulares
- `numpy`: operações numéricas
- `os`: Escrever arquivo csv a partir do df

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import os
print("Importações Concluídas")

## 1. Carregando os Dados

In [ ]:
# Carregamento dos dois datasets
df_clientes = pd.read_csv('../dados/Brutos/credito_aplicacao_clientes.csv', sep=',')
df_pedidos  = pd.read_csv('../dados/Brutos/credito_comportamental_pedidos.csv', sep=',')

print('Datasets carregados')
print(f'  Clientes: {df_clientes.shape[0]} linhas x {df_clientes.shape[1]} colunas')
print(f'  Pedidos:  {df_pedidos.shape[0]} linhas x {df_pedidos.shape[1]} colunas')

In [ ]:
df_clientes.head()

In [ ]:
df_pedidos.head()

In [ ]:
df_clientes_tratado = df_clientes.copy() #criando uma cópia do dataframe original para tratamento
df_pedidos_tratado  = df_pedidos.copy()

## 2. Tratamento de Dados por Tipo

Esta seção é responsável pelo tratamento do dataset para uma versão sem os problemas de campos nulos, não numéricos e intervalos.

### 2.1 Resolução de colunas numéricas dividas em Intervalos


#### 2.1.1 Identificação de Intervalos

In [ ]:
import re

# Padrão exato de intervalo numérico, ex.: (1000, 2500] ou [0, 5)
padrao_intervalo = re.compile(r"^\s*[\(\[]\s*-?\d+(?:[\.,]\d+)?\s*,\s*-?\d+(?:[\.,]\d+)?\s*[\)\]]\s*$")

colunas_intervalo = []

for coluna in df_clientes.columns:
    serie = df_clientes[coluna].dropna().astype(str).str.strip()
    if serie.empty:
        continue

    # Considera coluna de intervalo quando todos os valores não nulos seguem o padrão
    if serie.str.match(padrao_intervalo).all():
        colunas_intervalo.append(coluna)
        exemplos = serie.drop_duplicates().head(3).tolist()
        print(f"{coluna}: {exemplos}")

if not colunas_intervalo:
    print("Nenhuma coluna com padrão de intervalo foi encontrada.")
else:
    print("\nColunas identificadas:", colunas_intervalo)

#### 2.1.2 Transformação usando Mediana

Considerando este compormento e pelas análises feitas na **Fase 1**, será adotado como o uso da **mediana** como a a estratégia para transformar estas colunas de intervalos, do dataset de clientes, para valores numéricos. Pois pela quantidade de dados do dataset, usar a média poderia trazer dificuldades para discrepâncias de valores.

In [ ]:
# Conversao de intervalos para valor numerico usando a mediana da faixa (ponto medio)
intervalo_regex = re.compile(
    r"^\s*([\(\[])\s*(-?\d+(?:[\.,]\d+)?)\s*,\s*(-?\d+(?:[\.,]\d+)?)\s*([\)\]])\s*$"
 )

def intervalo_para_mediana(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()
    match = intervalo_regex.match(texto)
    if not match:
        return np.nan

    limite_inferior = float(match.group(2).replace(',', '.'))
    limite_superior = float(match.group(3).replace(',', '.'))

    # Mediana da faixa para intervalo continuo: ponto medio
    return (limite_inferior + limite_superior) / 2.0


colunas_convertidas = []
for coluna in colunas_intervalo:
    serie_convertida = df_clientes_tratado[coluna].apply(intervalo_para_mediana)

    # So converte se toda a parte nao nula foi interpretada corretamente
    if serie_convertida.notna().sum() == df_clientes_tratado[coluna].notna().sum():
        df_clientes_tratado[coluna] = serie_convertida
        colunas_convertidas.append(coluna)

print('Colunas convertidas com mediana da faixa:')
print(colunas_convertidas)

if colunas_convertidas:
    print('\nAmostra apos conversao:')
    print(df_clientes_tratado[colunas_convertidas].head())

In [ ]:
df_clientes_tratado.head()

Como é possível observar no head acima, agora o dataset de clientes tem os dados em intervalos transformado em apenas um mesmo dado numérico, utizando a mediana do intervalo.

### 2.2 Valores Nulos

#### 2.2.1 Idenficação e Observação

Assim como observado na **Seção 2.4 da Fase 1**, todas as colunas com valores nulos carregas informação, vamos observar quais eram

In [ ]:
# total de nulos e percentual em relação ao total de linhas
nulos = pd.DataFrame({
    'total_nulos': df_clientes_tratado.isnull().sum(),
    'pct_nulos': (df_clientes_tratado.isnull().sum() / len(df_clientes_tratado) * 100).round(1)
}).query('total_nulos > 0').sort_values('pct_nulos', ascending=False)

print('Colunas com valores nulos')
print(nulos.to_string())
print(f'\nTotal de colunas sem nenhum nulo: {df_clientes_tratado.shape[1] - len(nulos)} de {df_clientes_tratado.shape[1]}')

#### 2.2.2 Tratamento por Colunas de Valor Binário

Considerando essa realidade, a abordagem utilizada para este problema será **a criação de colunas binárias que identifiquem se há ou não valor naquele atributo**, exemplo: tem_ifood para identificar se o estabelecimento é cadastrado ou não no ifood.

In [ ]:
# Colunas com nulos no dataset tratado
colunas_com_nulos = df_clientes_tratado.columns[df_clientes_tratado.isna().any()].tolist()
print('Colunas com nulos:')
print(colunas_com_nulos)

def criar_coluna_presenca(df, nome_coluna, colunas_base):
    colunas_existentes = [c for c in colunas_base if c in df.columns]
    if not colunas_existentes:
        print(f"Nao foi possivel criar {nome_coluna}: colunas base nao encontradas.")
        return
    
    # 1 se existe pelo menos um valor preenchido nas colunas base; 0 caso contrario
    df[nome_coluna] = df[colunas_existentes].notna().any(axis=1).astype(int)
    print(f"{nome_coluna} criada a partir de: {colunas_existentes}")

# Indicadores solicitados
criar_coluna_presenca(
    df_clientes_tratado,
    'tem_google_maps',
    [
        'google_maps_avaliacao',
        'google_maps_contagem_avaliacoes',
        'google_maps_tem_website'
    ]
)

criar_coluna_presenca(
    df_clientes_tratado,
    'tem_ifood',
    [
        'ifood_contagem_avaliacoes',
        'ifood_faixa_preco'
    ]
)

criar_coluna_presenca(
    df_clientes_tratado,
    'tem_credores',
    ['serasa_credores']
)

print('\nResumo das novas colunas binarias:')
for col in ['tem_google_maps', 'tem_ifood', 'tem_credores']:
    if col in df_clientes_tratado.columns:
        print(f"{col}:\n{df_clientes_tratado[col].value_counts(dropna=False).sort_index()}\n")

df_clientes_tratado[['tem_google_maps', 'tem_ifood', 'tem_credores']].head()

#### 2.2.3 Preenchimento de Valores Nulos

Agora que temos uma coluna para diferenciar os campos que são nulos e os que são realmente zero, vamos preencher o dataset de valores nulos com 0, pois considerando o contexto dos dados, é a estratégia mais eficiente.

In [ ]:
# Preenchimento de nulos com 0 apos criacao das colunas de presenca
nulos_antes = int(df_clientes_tratado.isna().sum().sum())
print(f'Total de valores nulos antes: {nulos_antes}')

colunas_com_nulos_antes = df_clientes_tratado.columns[df_clientes_tratado.isna().any()].tolist()
print('Colunas com nulos antes do preenchimento:')
print(colunas_com_nulos_antes)

df_clientes_tratado = df_clientes_tratado.fillna(0)

nulos_depois = int(df_clientes_tratado.isna().sum().sum())
print(f'\nTotal de valores nulos depois: {nulos_depois}')
print('Preenchimento concluido.')

print('\nResumo rapido (primeiras linhas das colunas tratadas):')
print(df_clientes_tratado[colunas_com_nulos_antes].head())

In [ ]:
df_clientes_tratado.head()

### 2.3 Transformação de Variáveis Categóricas

Nesta subseção, as variáveis categóricas serão tratadas para se adequarem a colunas de valor numérico.

#### 2.3.1 Identificação das Variáveis Categóricas

Primeiro, vamos identificar quais colunas ainda são do tipo `object` (texto) e precisam de transformação.

In [ ]:
# Colunas categoricas restantes (object)
colunas_categoricas = df_clientes_tratado.select_dtypes(include='object').columns.tolist()

print(f'Colunas categoricas encontradas ({len(colunas_categoricas)}):')
for col in colunas_categoricas:
    n_unique = df_clientes_tratado[col].nunique()
    exemplos = df_clientes_tratado[col].dropna().unique()[:3].tolist()
    print(f'  {col}: {n_unique} valores únicos | ex: {exemplos}')

#### 2.3.2 `ifood_faixa_preco` — Escala Ordinal de Preço

A coluna `ifood_faixa_preco` contém valores do tipo `$`, `$$`, `$$$`, `$$$$`, `$$$$$`, que representam uma **escala ordinal** de preço. Será feito um mapeamento direto para inteiros de 1 a 5.

In [ ]:
# Mapeamento ordinal da faixa de preco iFood
mapa_faixa_preco = {
    0:   0,  # sem cadastro (preenchido com 0 anteriormente)
    '$': 1,
    '$$': 2,
    '$$$': 3,
    '$$$$': 4,
    '$$$$$': 5
}

# A coluna pode ter sido preenchida com 0 (int) ou string
df_clientes_tratado['ifood_faixa_preco'] = (
    df_clientes_tratado['ifood_faixa_preco']
    .astype(str)
    .str.strip()
    .map(lambda v: mapa_faixa_preco.get(v, mapa_faixa_preco.get(int(v) if v.lstrip('-').isdigit() else v, 0)))
)

print('Distribuicao de ifood_faixa_preco apos mapeamento:')
print(df_clientes_tratado['ifood_faixa_preco'].value_counts().sort_index())

#### 2.3.3 `natureza_juridica` — Label Encoding

A `natureza_juridica` é uma variável categórica nominal com 4 categorias. Será extraído apenas o **código numérico** presente no início da string (ex: `206-2`) para facilitar a modelagem.

In [ ]:
# Extração do código numérico da natureza jurídica
# Formato: "XXX-X - Descrição" → extrair "XXX-X" → converter para float substituindo '-' por '.'
df_clientes_tratado['natureza_juridica_cod'] = (
    df_clientes_tratado['natureza_juridica']
    .str.extract(r'^(\d{3}-\d)')
    .iloc[:, 0]
    .str.replace('-', '.', regex=False)
    .astype(float)
)

print('Distribuição de natureza_juridica_cod:')
print(df_clientes_tratado.groupby('natureza_juridica_cod')['natureza_juridica'].first())

# Remover coluna original de texto
df_clientes_tratado.drop(columns=['natureza_juridica'], inplace=True)
print('\nColuna natureza_juridica removida. Mantida apenas natureza_juridica_cod.')

#### 2.3.4 `fonte_cliente` — Label Encoding

A `fonte_cliente` possui categorias do tipo `Fonte 1`, `Fonte 2`, etc. Será extraído apenas o número da fonte.

In [ ]:
# Extração do número da fonte
df_clientes_tratado['fonte_cliente'] = (
    df_clientes_tratado['fonte_cliente']
    .str.extract(r'(\d+)')
    .iloc[:, 0]
    .astype(int)
)

print('Distribuição de fonte_cliente:')
print(df_clientes_tratado['fonte_cliente'].value_counts().sort_index())

#### 2.3.5 `uf` e `municipio` — One-Hot Encoding

As colunas `uf` e `municipio` são variáveis nominais geográficas. Como possuem poucas categorias (2 UFs e 18 municípios), será aplicado **One-Hot Encoding** para evitar ordenamento artificial.

In [ ]:
# One-Hot Encoding para uf e municipio
for coluna in ['uf', 'municipio']:
    dummies = pd.get_dummies(df_clientes_tratado[coluna], prefix=coluna, drop_first=True, dtype=int)
    df_clientes_tratado = pd.concat([df_clientes_tratado, dummies], axis=1)
    df_clientes_tratado.drop(columns=[coluna], inplace=True)
    print(f'{coluna}: {len(dummies.columns)} colunas criadas → {dummies.columns.tolist()[:5]}...')

print(f'\nShape após OHE: {df_clientes_tratado.shape}')

#### 2.3.6 `segmento_cliente` — Label Encoding

O `segmento_cliente` tem 18 categorias representando tipos de estabelecimento. Será aplicado **Label Encoding**, pois o CNAE já captura essa informação de forma mais granular (tratado na Seção 3.2). Esta coluna serve como um atalho semântico.

In [ ]:
le_segmento = LabelEncoder()
df_clientes_tratado['segmento_cliente'] = le_segmento.fit_transform(
    df_clientes_tratado['segmento_cliente'].astype(str)
)

print('Mapeamento segmento_cliente → inteiro (primeiros 8):')
for i, cls in enumerate(le_segmento.classes_[:8]):
    print(f'  {i}: {cls}')

print(f'\nDistribuição:')
print(df_clientes_tratado['segmento_cliente'].value_counts().sort_index())

#### 2.3.7 Verificação Final — Nenhum Objeto Restante

In [ ]:
# Verifica se ainda existem colunas do tipo object (exceto cnae e cnae_descricao, tratadas na seção 3)
restantes = df_clientes_tratado.select_dtypes(include='object').columns.tolist()

if restantes:
    print(f'Colunas object ainda presentes (serão tratadas nas próximas seções): {restantes}')
else:
    print('Todas as colunas categóricas foram convertidas para numérico.')

print(f'\nShape atual: {df_clientes_tratado.shape}')
df_clientes_tratado.dtypes

## 3. Tratamento de Dados Complexos

Nesta seção será tratado colunas com valores complexos, como listas e códigos


### 3.1 `serasa_credores` — Valores Compostos (Lista)

A coluna `serasa_credores` contém listas de setores credores separados por vírgula (ex: `"Financeiro, Bancos e Financeiro, Financeiro"`). Como os valores nulos já foram substituídos por `0`, faremos a explosão em colunas binárias por setor credor.

In [ ]:
# Analise dos setores credores presentes
credores_raw = df_clientes['serasa_credores'].dropna()

# Expandir todos os valores separados por vírgula
todos_credores = (
    credores_raw
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
)

print('Setores credores encontrados (com contagem):')
print(todos_credores)

In [ ]:
# Criar colunas binárias por setor credor (multi-label)
setores_unicos = todos_credores.index.tolist()

def tem_credor(valor_bruto, setor):
    """Retorna 1 se o setor está presente na lista de credores do cliente."""
    if pd.isna(valor_bruto) or str(valor_bruto).strip() == '0':
        return 0
    credores_lista = [c.strip() for c in str(valor_bruto).split(',')]
    return 1 if setor in credores_lista else 0

# Usar o dataset original para extrair os credores (antes do fillna(0))
credores_originais = df_clientes['serasa_credores'].copy()

for setor in setores_unicos:
    nome_col = 'credor_' + setor.lower().replace(' ', '_').replace('&', 'e')
    df_clientes_tratado[nome_col] = credores_originais.apply(lambda v: tem_credor(v, setor))

colunas_credor = [c for c in df_clientes_tratado.columns if c.startswith('credor_')]
print(f'Colunas criadas: {colunas_credor}')
print(f'\nSoma por setor credor:')
print(df_clientes_tratado[colunas_credor].sum().sort_values(ascending=False))

# Remover coluna original de texto (serasa_credores já foi substituída por 0 no fillna)
# A coluna 'serasa_credores' nesse ponto contém 0, então a removemos
df_clientes_tratado.drop(columns=['serasa_credores'], inplace=True)
print('\nColuna serasa_credores removida — substituída pelas colunas binárias por setor.')

### 3.2 `cnae_codigo` — Extração de Seção e Divisão CNAE

O código CNAE tem o formato `DD.DD-D-DD` onde os **dois primeiros dígitos representam a divisão** e os **dois primeiros dígitos agrupados por faixa representam a seção** (ex: 56 → Alimentação, 55 → Alojamento, 10 → Indústria de Alimentos).

Extrairemos:
- `cnae_divisao`: os dois primeiros dígitos numéricos (granularidade média)
- `cnae_secao`: seção da atividade econômica (nível mais alto de agregação)

In [ ]:
# Visualizando os códigos CNAE presentes
print('Distribuição dos 2 primeiros dígitos do CNAE (divisão):')
print(df_clientes['cnae_codigo'].str[:2].value_counts().sort_values(ascending=False))

print('\nExemplos de cnae_codigo e cnae_descricao:')
print(df_clientes[['cnae_codigo', 'cnae_descricao']].drop_duplicates().head(10))

In [ ]:
# Extração da divisão CNAE (2 primeiros dígitos)
df_clientes_tratado['cnae_divisao'] = (
    df_clientes_tratado['cnae_codigo']
    .str.extract(r'^(\d{2})')
    .iloc[:, 0]
    .astype(int)
)

# Mapeamento da Seção CNAE a partir da divisão
# Baseado na estrutura oficial do CNAE 2.0
def divisao_para_secao(divisao):
    """Retorna a letra da Seção CNAE baseado na divisão (int)."""
    if 1  <= divisao <= 3:   return 'A'  # Agricultura
    if 5  <= divisao <= 9:   return 'B'  # Indústrias Extrativas
    if 10 <= divisao <= 33:  return 'C'  # Indústrias de Transformação
    if divisao == 35:        return 'D'  # Eletricidade e Gás
    if 36 <= divisao <= 39:  return 'E'  # Água e Saneamento
    if 41 <= divisao <= 43:  return 'F'  # Construção
    if 45 <= divisao <= 47:  return 'G'  # Comércio e Reparação
    if 49 <= divisao <= 53:  return 'H'  # Transporte
    if 55 <= divisao <= 56:  return 'I'  # Alojamento e Alimentação
    if 58 <= divisao <= 63:  return 'J'  # Informação e Comunicação
    if 64 <= divisao <= 66:  return 'K'  # Financeiro
    if divisao == 68:        return 'L'  # Imobiliário
    if 69 <= divisao <= 75:  return 'M'  # Atividades Profissionais
    if 77 <= divisao <= 82:  return 'N'  # Atividades Administrativas
    if divisao == 84:        return 'O'  # Administração Pública
    if divisao == 85:        return 'P'  # Educação
    if 86 <= divisao <= 88:  return 'Q'  # Saúde
    if 90 <= divisao <= 93:  return 'R'  # Artes e Cultura
    if 94 <= divisao <= 96:  return 'S'  # Outras Atividades de Serviço
    if 97 <= divisao <= 98:  return 'T'  # Domésticos
    if divisao == 99:        return 'U'  # Organismos Internacionais
    return 'Outro'

df_clientes_tratado['cnae_secao'] = df_clientes_tratado['cnae_divisao'].apply(divisao_para_secao)

print('Distribuição por Seção CNAE:')
print(df_clientes_tratado['cnae_secao'].value_counts())

print('\nDistribuição por Divisão CNAE:')
print(df_clientes_tratado['cnae_divisao'].value_counts().sort_index())

In [ ]:
# Label Encoding para cnae_secao
le_secao = LabelEncoder()
df_clientes_tratado['cnae_secao'] = le_secao.fit_transform(
    df_clientes_tratado['cnae_secao']
)

print('Mapeamento cnae_secao:')
for i, cls in enumerate(le_secao.classes_):
    print(f'  {i}: Seção {cls}')

# Remover colunas originais de texto do CNAE
df_clientes_tratado.drop(columns=['cnae_codigo', 'cnae_descricao'], inplace=True)
print('\nColunas cnae_codigo e cnae_descricao removidas — substituídas por cnae_divisao e cnae_secao.')

## 4. Agregação e Criação de Colunas

Nesta seção há o processo de agregação e criação de colunas do dataset de Clientes a partir dos insights sobre possíveis features importantes

### 4.1 Merge com Dados de Pedidos (Feature Engineering Comportamental)

Os dados do dataset de pedidos oferecem informações valiosas sobre o comportamento de pagamento dos clientes. Vamos criar features agregadas por cliente a partir de `df_pedidos`.

In [ ]:
# Agregações por cliente a partir do histórico de pedidos
agg_pedidos = df_pedidos_tratado.groupby('id_cliente').agg(
    qtd_pedidos=('id_pedido', 'count'),
    valor_total_pedidos=('valor', 'sum'),
    valor_medio_pedido=('valor', 'mean'),
    valor_max_pedido=('valor', 'max'),
    atraso_total=('atraso', 'sum'),
    atraso_medio=('atraso', 'mean'),
    atraso_max=('atraso', 'max'),
    pedidos_com_atraso=('atraso', lambda x: (x > 0).sum()),
).reset_index()

# Taxa de atraso: proporção de pedidos com atraso
agg_pedidos['taxa_atraso'] = (
    agg_pedidos['pedidos_com_atraso'] / agg_pedidos['qtd_pedidos']
).round(4)

print('Agregações criadas:')
print(agg_pedidos.columns.tolist())
print(f'\nClientes com histórico de pedidos: {len(agg_pedidos)}')
agg_pedidos.head()

In [ ]:
# Merge left: mantém todos os clientes, preenche com 0 quem não tem pedidos
df_clientes_tratado = df_clientes_tratado.merge(
    agg_pedidos,
    on='id_cliente',
    how='left'
)

colunas_pedido = agg_pedidos.columns.drop('id_cliente').tolist()
df_clientes_tratado[colunas_pedido] = df_clientes_tratado[colunas_pedido].fillna(0)

print(f'Shape após merge: {df_clientes_tratado.shape}')
print(f'Clientes sem histórico de pedidos (qtd_pedidos == 0): {(df_clientes_tratado["qtd_pedidos"] == 0).sum()}')

### 4.2 Feature Engineering — Colunas Derivadas

Com base nas variáveis disponíveis, serão criadas features derivadas que capturam relações entre os atributos dos clientes e são potencialmente preditivas para inadimplência.

In [ ]:
# ── Feature 1: Score de negativação Serasa ────────────────────────────────────
# Combina contagem de negativações, protestos e presença de sócio negativado
df_clientes_tratado['score_serasa_negativo'] = (
    df_clientes_tratado['serasa_contagem_negativacoes'] +
    df_clientes_tratado['serasa_contagem_protestos'] +
    df_clientes_tratado['serasa_socio_tem_negativacao']
)

# ── Feature 2: Índice de presença digital ─────────────────────────────────────
# Soma das presença no iFood e Google Maps (0, 1 ou 2)
df_clientes_tratado['presenca_digital'] = (
    df_clientes_tratado['tem_ifood'] +
    df_clientes_tratado['tem_google_maps']
)

# ── Feature 3: Capital social por ano de empresa ──────────────────────────────
# Proxy de solidez financeira relativa ao tempo
df_clientes_tratado['capital_por_ano'] = (
    df_clientes_tratado['capital_social'] /
    (df_clientes_tratado['idade_cnpj'].replace(0, np.nan) / 365)
).fillna(0).round(2)

# ── Feature 4: Tem histórico de atraso ────────────────────────────────────────
# Indica se o cliente já atrasou algum pagamento (binário)
df_clientes_tratado['tem_historico_atraso'] = (
    df_clientes_tratado['atraso_total'] > 0
).astype(int)

# ── Feature 5: Valor médio por pedido normalizado ─────────────────────────────
# Categoria de ticket médio
df_clientes_tratado['ticket_medio_alto'] = (
    df_clientes_tratado['valor_medio_pedido'] > df_clientes_tratado['valor_medio_pedido'].median()
).astype(int)

# ── Feature 6: Reputação Google Maps ──────────────────────────────────────────
# Combina nota média e quantidade de avaliações em um único índice
df_clientes_tratado['reputacao_google'] = (
    df_clientes_tratado['google_maps_avaliacao'] *
    np.log1p(df_clientes_tratado['google_maps_contagem_avaliacoes'])
).round(4)

print('Features criadas com sucesso:')
features_novas = ['score_serasa_negativo', 'presenca_digital', 'capital_por_ano',
                  'tem_historico_atraso', 'ticket_medio_alto', 'reputacao_google']
print(df_clientes_tratado[features_novas].describe().round(2))

### 4.3 Análise de Outliers

Verificamos a interferência de outliers nas colunas numéricas contínuas mais relevantes, usando o critério **IQR (Interquartile Range)**.

In [ ]:
# Colunas numericas continuas para analise de outliers
colunas_numericas = [
    'capital_social', 'idade_cnpj', 'serasa_contagem_negativacoes',
    'ifood_contagem_avaliacoes', 'google_maps_avaliacao',
    'google_maps_contagem_avaliacoes', 'valor_total_pedidos',
    'valor_medio_pedido', 'atraso_total', 'capital_por_ano'
]

# Filtra apenas as que existem no df
colunas_numericas = [c for c in colunas_numericas if c in df_clientes_tratado.columns]

resultados_outlier = []
for col in colunas_numericas:
    serie = df_clientes_tratado[col].dropna()
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    n_outliers = ((serie < limite_inf) | (serie > limite_sup)).sum()
    pct = round(n_outliers / len(serie) * 100, 1)
    resultados_outlier.append({'coluna': col, 'Q1': round(Q1, 2), 'Q3': round(Q3, 2),
                               'IQR': round(IQR, 2), 'lim_inf': round(limite_inf, 2),
                               'lim_sup': round(limite_sup, 2), 'n_outliers': n_outliers, 'pct': pct})

df_outliers = pd.DataFrame(resultados_outlier).sort_values('n_outliers', ascending=False)
print('Resumo de outliers por coluna (critério IQR):')
print(df_outliers[['coluna', 'n_outliers', 'pct', 'lim_sup']].to_string(index=False))

In [ ]:
# Visualização dos boxplots das colunas com mais outliers
top_outlier_cols = df_outliers[df_outliers['n_outliers'] > 0]['coluna'].tolist()[:6]

if top_outlier_cols:
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()
    for i, col in enumerate(top_outlier_cols):
        axes[i].boxplot(df_clientes_tratado[col].dropna(), vert=True, patch_artist=True,
                        boxprops=dict(facecolor='steelblue', alpha=0.7))
        axes[i].set_title(col, fontsize=9)
        axes[i].set_xticks([])
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    plt.suptitle('Boxplots — Colunas com Outliers', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Nenhuma coluna com outliers expressivos encontrada.')

#### Decisão sobre Outliers

Dado o contexto de negócio (crédito B2B com 200 registros), **não serão removidos outliers** nesta etapa pelos seguintes motivos:

1. **Dataset pequeno (200 linhas)**: remoção de outliers reduziria ainda mais os dados disponíveis para treino.
2. **Outliers são informativos**: um `capital_social` muito alto ou `serasa_contagem_negativacoes` elevada pode ser exatamente o sinal discriminativo para prever inadimplência.
3. **Normalização posterior**: a padronização (seção 4.4) já minimiza o impacto dos outliers em algoritmos sensíveis à escala.
4. **Modelos robustos**: Random Forest e XGBoost são naturalmente resistentes a outliers.

### 4.4 Avaliação de Colunas com Baixa Variância

Colunas com variância muito baixa (quase constantes) não contribuem para a separação das classes e podem prejudicar o modelo.

In [ ]:
# Calcular variância das colunas numéricas
df_num = df_clientes_tratado.select_dtypes(include=[np.number]).drop(columns=['id_cliente', 'inadimplente'], errors='ignore')

variancia = df_num.var().sort_values()

# Threshold: colunas com variância < 0.01 são candidatas a remoção
LIMIAR_VARIANCIA = 0.01
colunas_baixa_variancia = variancia[variancia < LIMIAR_VARIANCIA].index.tolist()

print(f'Colunas com variância < {LIMIAR_VARIANCIA}:')
if colunas_baixa_variancia:
    print(variancia[variancia < LIMIAR_VARIANCIA].to_string())
else:
    print('Nenhuma coluna com variância abaixo do limiar encontrada.')

print(f'\nVariância das 10 colunas com menor variância:')
print(variancia.head(10).round(6))

In [ ]:
# Remover colunas com variância abaixo do limiar (se existirem)
if colunas_baixa_variancia:
    # Verificar se são realmente pouco informativas
    print('Análise das colunas candidatas à remoção:')
    for col in colunas_baixa_variancia:
        print(f'  {col}: var={variancia[col]:.6f}, unique={df_clientes_tratado[col].nunique()}, values={df_clientes_tratado[col].unique()[:5]}')
    
    df_clientes_tratado.drop(columns=colunas_baixa_variancia, inplace=True)
    print(f'\n{len(colunas_baixa_variancia)} coluna(s) removida(s): {colunas_baixa_variancia}')
else:
    print('Nenhuma coluna removida por baixa variância.')

print(f'\nShape atual: {df_clientes_tratado.shape}')

### 4.5 Separação Treino/Teste

A separação entre treino e teste deve ser feita **antes** de transformações críticas como normalização/padronização para evitar **data leakage** — ou seja, o modelo "vazando" informações do conjunto de teste durante o ajuste dos transformadores.

Estratégia: **80% treino / 20% teste** com `stratify` na variável alvo `inadimplente` para manter a proporção da classe minoritária.

In [ ]:
# Separacao Treino/Teste com stratify
RANDOM_STATE = 42
PROPORCAO_TESTE = 0.2

# Colunas de features e target
colunas_excluir = ['id_cliente', 'inadimplente']
X = df_clientes_tratado.drop(columns=colunas_excluir, errors='ignore')
y = df_clientes_tratado['inadimplente']

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=PROPORCAO_TESTE,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Separação treino/teste concluída:')
print(f'  Treino: {X_treino.shape[0]} amostras ({len(X_treino)/len(X)*100:.0f}%)')
print(f'  Teste:  {X_teste.shape[0]} amostras ({len(X_teste)/len(X)*100:.0f}%)')
print(f'\nDistribuição da classe alvo (inadimplente):')
print(f'  Treino — 0: {(y_treino==0).sum()}, 1: {(y_treino==1).sum()} '
      f'(taxa: {y_treino.mean()*100:.1f}%)')
print(f'  Teste  — 0: {(y_teste==0).sum()}, 1: {(y_teste==1).sum()} '
      f'(taxa: {y_teste.mean()*100:.1f}%)')

### 4.6 Padronização/Normalização das Variáveis Numéricas

A padronização é feita **apenas no conjunto de treino** para ajustar o scaler. O mesmo scaler é então aplicado ao conjunto de teste, evitando leakage.

Usaremos **`StandardScaler`** (média 0, desvio padrão 1), pois:
- Adequado para modelos lineares e SVM
- Random Forest e XGBoost não precisam, mas não são prejudicados

In [ ]:
# Identificar colunas numericas continuas para escalar (excluir binárias e dummies)
# Heurística: colunas com mais de 2 valores únicos e não binárias
colunas_binarias = [c for c in X_treino.columns if X_treino[c].nunique() <= 2]
colunas_para_escalar = [c for c in X_treino.select_dtypes(include=[np.number]).columns
                        if c not in colunas_binarias]

print(f'Colunas binárias (não serão escaladas): {len(colunas_binarias)}')
print(f'Colunas a escalar: {len(colunas_para_escalar)}')
print(colunas_para_escalar)

In [ ]:
# Ajustar scaler APENAS no treino — aplicar em treino e teste
scaler = StandardScaler()

X_treino_scaled = X_treino.copy()
X_teste_scaled  = X_teste.copy()

if colunas_para_escalar:
    X_treino_scaled[colunas_para_escalar] = scaler.fit_transform(X_treino[colunas_para_escalar])
    X_teste_scaled[colunas_para_escalar]  = scaler.transform(X_teste[colunas_para_escalar])

print('Padronização aplicada com StandardScaler.')
print('\nEstatísticas do treino após escalonamento (deve ter média ≈ 0):')
print(X_treino_scaled[colunas_para_escalar[:5]].describe().round(3))

## 5. Salvando o Novo Dataset

Nesta seção será salvo o novo dataset, além de explicitar a interpretação por cada nova coluna adicionada

### 5.1 Salvamento do Dataset Completo Tratado

In [ ]:
# Pasta e nome do arquivo de saida
pasta_saida = '../dados/tratados'
os.makedirs(pasta_saida, exist_ok=True)

# Dataset completo tratado (sem divisão)
nome_completo = 'clientes_preprocessado.csv'
df_clientes_tratado.to_csv(os.path.join(pasta_saida, nome_completo), index=False, encoding='utf-8-sig')
print(f'Dataset completo salvo: {nome_completo}')
print(f'  Dimensão: {df_clientes_tratado.shape[0]} linhas x {df_clientes_tratado.shape[1]} colunas')

# Conjuntos de treino e teste (com escalonamento)
df_treino_final = X_treino_scaled.copy()
df_treino_final['inadimplente'] = y_treino.values
df_treino_final.to_csv(os.path.join(pasta_saida, 'treino_preprocessado.csv'), index=False, encoding='utf-8-sig')

df_teste_final = X_teste_scaled.copy()
df_teste_final['inadimplente'] = y_teste.values
df_teste_final.to_csv(os.path.join(pasta_saida, 'teste_preprocessado.csv'), index=False, encoding='utf-8-sig')

print(f'\nConjunto de treino salvo: treino_preprocessado.csv ({df_treino_final.shape})')
print(f'Conjunto de teste salvo:  teste_preprocessado.csv ({df_teste_final.shape})')

In [ ]:
df_clientes_tratado.info()

## 6. Resumo Geral das Atividades

In [ ]:
print('=' * 65)
print('RESUMO — FASE 2: PRÉ-PROCESSAMENTO')
print('=' * 65)

print('\n✅ 2.1 Intervalos → Mediana do intervalo')
print('   Colunas tratadas: capital_social, idade_cnpj, ifood_contagem_avaliacoes,')
print('   google_maps_avaliacao, google_maps_contagem_avaliacoes')

print('\n✅ 2.2 Valores Nulos')
print('   Colunas de presença criadas: tem_google_maps, tem_ifood, tem_credores')
print('   Nulos preenchidos com 0 (ausência de dado = não tem cadastro)')

print('\n✅ 2.3 Variáveis Categóricas → Numéricas')
print('   ifood_faixa_preco: mapeamento ordinal ($=1 a $$$$$=5)')
print('   natureza_juridica: código numérico extraído (natureza_juridica_cod)')
print('   fonte_cliente: número extraído (1–5)')
print('   uf e municipio: One-Hot Encoding (drop_first=True)')
print('   segmento_cliente: Label Encoding')

print('\n✅ 3.1 serasa_credores (lista composta) → Colunas binárias por setor')
print('   Exemplo: credor_tecnologia, credor_financeiro, credor_saúde...')

print('\n✅ 3.2 CNAE → cnae_divisao (int) + cnae_secao (label encoded)')
print('   Seção CNAE identifica o grande setor econômico do estabelecimento')

print('\n✅ 4.1 Merge com dados de pedidos')
print('   Features: qtd_pedidos, valor_total, valor_medio, atraso_total, taxa_atraso...')

print('\n✅ 4.2 Feature Engineering')
print('   score_serasa_negativo, presenca_digital, capital_por_ano,')
print('   tem_historico_atraso, ticket_medio_alto, reputacao_google')

print('\n✅ 4.3 Outliers analisados via IQR → mantidos (dataset pequeno + modelos robustos)')

print('\n✅ 4.4 Baixa variância avaliada → colunas quase constantes removidas')

print('\n✅ 4.5 Separação Treino/Teste (80/20, stratify, random_state=42)')

print('\n✅ 4.6 StandardScaler ajustado no treino, aplicado no teste (sem leakage)')

print('\n✅ 5. Arquivos salvos em dados/tratados/')
print('   clientes_preprocessado.csv    → dataset completo tratado')
print('   treino_preprocessado.csv      → treino escalado + target')
print('   teste_preprocessado.csv       → teste escalado + target')

print('\n' + '=' * 65)
print(f'Shape final do dataset tratado: {df_clientes_tratado.shape}')
print(f'Features para modelagem: {X_treino_scaled.shape[1]}')
print(f'Inadimplência na base: {y.mean()*100:.1f}%')
print('=' * 65)